In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only ".
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory



# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

## Imports

In [2]:
!pip install transformers

In [3]:
from transformers import AutoModelForAudioClassification,AutoFeatureExtractor
from torch.utils.data import Dataset,DataLoader
import os
import torch.nn as nn
import torch
from sklearn.model_selection import train_test_split
import librosa
import wandb
from kaggle_secrets import UserSecretsClient
from sklearn.metrics import f1_score

/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

In [4]:
user_secrets = UserSecretsClient()
wandb_api = user_secrets.get_secret("login")
os.environ["WANDB_API_KEY"]=wandb_api
wandb.login()
print("Wandb login successful")

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: Currently logged in as: 23f1000559 (23f1000559-indian-institute-of-technology-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Wandb login successful


## Model

In [5]:
model_name = "MIT/ast-finetuned-audioset-10-10-0.4593"

In [6]:
model = AutoModelForAudioClassification.from_pretrained(model_name)
feature_extractor = AutoFeatureExtractor.from_pretrained(model_name)



config.json: 0.00B [00:00, ?B/s]

2026-03-07 07:57:59.091193: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772870279.293010      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772870279.355802      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772870279.885227      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772870279.885267      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772870279.885271      55 computation_placer.cc:177] computation placer alr

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

## Load Dataset

In [7]:
!ls /kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems/blues/blues.00000

bass.wav  drums.wav  other.wav	vocals.wav


In [10]:
# collect song directories for mixing stems
song_dirs=[]
labels = []
genres = ["blues","classical","country","disco","hiphop","jazz","metal","pop","reggae","rock"]
root_dir = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems"
for genre_idx, genre in enumerate(genres):
    genre_path = os.path.join(root_dir, genre)
    for song in os.listdir(genre_path):
        song_path = os.path.join(genre_path, song)
        song_dirs.append(song_path)
        labels.append(genre_idx)

In [13]:
print(type(song_dirs))
print(type(labels))

<class 'list'>
<class 'list'>


In [14]:
#fix length to 10 s for this model
max_len = 160000
def pad_truncate(audio):
    if len(audio) > max_len:
        audio = audio[:max_len]
    else:
        pad_len = max_len - len(audio)
        audio = np.pad(audio, (0, pad_len))
    return audio

In [15]:
# feature extracting 
features=[]
labels_list = []
stems = ["bass.wav","drums.wav","vocals.wav","other.wav"]
for song_dir,label in zip(song_dirs,labels):
    stem_audio =[]
    for stem in stems:
        stem_path = os.path.join(song_dir,stem)
        audio,sr = librosa.load(stem_path,sr=16000)
        audio = pad_truncate(audio)
        stem_audio.append(audio)
    # mix stems
    mix = np.sum(stem_audio,axis=0) / len(stem_audio)
    #normalising
    mix = mix / np.max(np.abs(mix) + 1e-6)
    inputs = feature_extractor(
        mix, sampling_rate=16000,return_tensors="pt"
    )
    features.append(inputs["input_values"].squeeze(0))
    labels_list.append(label)

In [16]:
print(type(song_dirs))
print(type(labels))

<class 'list'>
<class 'list'>


In [17]:
features = torch.stack(features)
labels_tensor = torch.tensor(labels_list)

In [18]:
features.shape

torch.Size([1000, 1024, 128])

In [19]:
# train val split
train_files,val_files,train_labels,val_labels = train_test_split(features,labels_tensor,random_state=42,test_size=0.2,stratify=labels_tensor)

In [20]:
class FeatureDataset(Dataset):
    def __init__(self,x,y):
        self.x=x
        self.y=y
    def __len__(self):
        return len(self.x)
    def __getitem__(self,idx):
        return self.x[idx],self.y[idx]

In [21]:
# dataset instance 
train_dataset = FeatureDataset(train_files,train_labels)
val_dataset = FeatureDataset(val_files,val_labels)

In [22]:
#dataloaders
train_loader = DataLoader(train_dataset,batch_size=4,shuffle=True)
val_loader = DataLoader(val_dataset,batch_size=4,shuffle=False)

## Training Loop

In [23]:
# loss and optimisers
criterion = nn.CrossEntropyLoss()
optim = torch.optim.AdamW(model.parameters(),lr=1e-4)

In [24]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [25]:
# # Start a new wandb run 
# run = wandb.init(
#     # Set the wandb project where this run will be logged.
#     project="23f1000559-t12026",
#     entity ="23f1000559-indian-institute-of-technology-madras",
#     # Track hyperparameters and run metadata.
#     config={
#         "epochs": 2,
#         "batch_size": 4,
#         "learning_rate": 1e-4,
#         "model": "MIT/ast-finetuned-audioset"
#     },
# )

In [26]:
model.to(device)
num_epochs = 2
for epoch in range(num_epochs):
    total_loss = 0
    model.train()
    for x,y in train_loader:
        x = x.to(device)
        y = y.to(device)
        output = model(x)
        logits = output.logits
        loss = criterion(logits,y)
        optim.zero_grad()
        loss.backward()
        optim.step()
        total_loss +=loss.item()
    avg_loss = loss/len(train_loader)
    #Log loss to wandb
    # wandb.log({"epoch": epoch+1, "train_loss": avg_loss})
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}")
    

Epoch 1/2, Loss: 0.0055
Epoch 2/2, Loss: 0.0036


## Validation Loop

In [27]:
model.eval()
total_val_loss =0
all_preds=[]
all_labels = []
with torch.no_grad():
    for x_val,y_val in val_loader:
        x_val = x_val.to(device)
        y_val = y_val.to(device)
        output= model(x_val)
        logits = output.logits
        loss = criterion(logits, y_val)
        total_val_loss += loss.item()
        preds = logits.argmax(dim=1)
        # store for f1
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y_val.cpu().numpy())
avg_val_loss = total_val_loss / len(val_loader)
val_macro_f1 = f1_score(all_labels,all_preds,average="macro")
print(f"Val Loss: {avg_val_loss:.4f}, Val Macro F1: {val_macro_f1:.4f}")
# log to wandb
# wandb.log({"val_loss": avg_val_loss, "val_macro_f1": val_macro_f1})
        

Val Loss: 0.8131, Val Macro F1: 0.6670


## Submission

In [29]:
genres = ["blues","classical","country","disco","hiphop",
          "jazz","metal","pop","reggae","rock"]
idx_genre = {i: g for i , g in enumerate(genres)}

In [30]:
test_df = pd.read_csv("/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/test.csv")

In [31]:
max_len = 160000
def pad_truncate(audio, max_len=max_len):
    if len(audio) > max_len:
        audio = audio[:max_len]
    else:
        pad_len = max_len - len(audio)
        audio = np.pad(audio, (0, pad_len))
    return audio
def predict_genre(file_path, model, feature_extractor, device):
    audio, sr = librosa.load(file_path, sr=16000)
    audio = pad_truncate(audio)
    
    inputs = feature_extractor(audio, sampling_rate=16000, return_tensors="pt")
    input_values = inputs["input_values"].to(device)
    with torch.no_grad():
        output = model(input_values)
        logits = output.logits
        pred_idx = logits.argmax(dim=1).item()
    return idx_genre[pred_idx]

In [34]:
predictions = []
model.eval()
model.to(device)
for file_path in test_df["filename"]:
    root = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup"
    final_path = os.path.join(root,file_path)
    genre_pred = predict_genre(final_path, model, feature_extractor, device)
    predictions.append(genre_pred)

In [35]:
submission = pd.DataFrame({
    "id": test_df["id"],
    "genre": predictions
})
submission.to_csv("submission.csv", index=False)
print(submission.head())

   id      genre
0   1        pop
1   2  classical
2   3      disco
3   4       rock
4   5    country


In [ ]:
# /kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/mashups/song0001.wav
# mashups/song0001.wav